# 🌍 **Lumeria - Egypt Tourism Popularity-Based Recommendation System** 🇪🇬

> **Note:** This project implements a Popularity-Based Recommendation System using the Bayesian Average (Weighted Rating) formula to accurately suggest top-rated Egyptian tourist attractions for the Lumeria mobile application.

<div align="center">
  <img src="https://pplx-res.cloudinary.com/image/upload/pplx_search_images/3d5d97094451768c8a00c422700eb0b6b2e3dc78.jpg" width="600">
</div>

---

# 🚨 **Project Overview**

This project applies **Popularity-Based Filtering** to build a lightweight, fast, and highly effective recommendation engine. By analyzing real user ratings and the volume of interactions, the system successfully highlights the most popular and genuinely highly-rated Egyptian historical and recreational sites, optimized for seamless integration into the Lumeria tourism application.

---

# **🔍 Data Preparation & Optimization**

- **Curated Datasets:** Integrated two main datasets: a comprehensive list of Egyptian places and a dataset of user ratings to ensure reliable recommendations.
- **Data Cleaning:** Handled missing values and removed duplicate records to maintain high data integrity.
- **Aggregation:** Grouped ratings by location to accurately calculate the total vote count and average rating for each specific place.

---

# 🛠 **Technical Implementation**

**Core Technologies**
- **Framework:** Python & Pandas (Data Manipulation), FastAPI (Deployment).
- **Algorithm:** Bayesian Average Formula (True Bayesian Estimate).
- **Optimization:** Applied an 80th percentile threshold to filter out places with insufficient ratings, preventing newly added or rarely visited places from skewing the top recommendations.
- **Resources:**
  - [**Lumeria Kaggle Dataset**](https://www.kaggle.com/datasets/mernaabass/egypt-tourism-places-and-ratings)

---

# **⚙️ System Workflow**

1.  **Data Ingestion:** Loading `Places.csv` and `Users.csv` datasets.
2.  **Merging & Cleansing:** Merging datasets on place IDs and filtering out unrated locations.
3.  **Metric Calculation:** Computing the global mean rating ($C$) and the minimum votes required ($m$).
4.  **Applying the Formula:** Calculating the Bayesian Average Weighted Rating (WR) for each qualifying place using: $WR = \left(\frac{v}{v+m}\right)R + \left(\frac{m}{v+m}\right)C$.
5.  **Location Filtering:** Creating dynamic recommendations based on specific cities (e.g., Top places in Cairo vs. Luxor).

---

# **📁 Dataset Specifications**

- **Source:** Lumeria Tourism Dataset (Kaggle).
- **Composition:** 161 unique tourist sites and 1,038 user rating interactions.
- **Key Features:** Location, Type of Site, User ID, and Rating (1-5 scale).
- **Performance:** Successfully identifies and ranks the absolute top places, balancing both the quantity of visits and the quality of user reviews.

---

# **🚀 Implementation Roadmap**

**Phase 1: Data Engineering & EDA**
- ✅ Uploaded and linked datasets on Kaggle.
- ✅ Performed Exploratory Data Analysis (EDA) on top locations.
- ✅ Handled data duplication (ensuring exactly 161 unique places) and null values.

**Phase 2: Recommendation Engine Build**
- ✅ Calculated base metrics (Average Rating & Count).
- ✅ Successfully applied the Bayesian Average algorithm.

**Phase 3: Deployment Preparation**
- ✅ Packaged the model into a dynamic function supporting city-based filtering.
- ✅ **Next Step:** Deploying the engine as a RESTful API on Hugging Face Spaces to feed the Lumeria Flutter App.

# **1. IMPORT LIBRARIES**
Importing the essential libraries for data manipulation and visualization.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt 

import warnings
warnings.filterwarnings('ignore')

# **2. LOAD DATASETS**
Loading the Lumeria places and user ratings datasets.

In [ ]:
places_df = pd.read_csv("/kaggle/input/datasets/mernaabass/egypt-tourism-places-and-ratings/Places.csv")
users_df = pd.read_csv("/kaggle/input/datasets/mernaabass/egypt-tourism-places-and-ratings/Users.csv")

In [ ]:
places_df.head()

In [ ]:
users_df.head()

# **3. DATA MERGING & SANITY CHECKS**
Merging the datasets, removing duplicates, handling missing values, and validating the final shape.

In [ ]:
places_df = places_df.drop_duplicates(subset=['Name'], keep='first')

In [ ]:
df = pd.merge(places_df, users_df, left_on="ID", right_on="place_id", how="left")
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
total_places = places_df.shape[0]
rated_places = df['Name'].nunique()

print(f"Total Places in Lumeria Database: {total_places}")
print(f"Places that received at least one rating: {rated_places}")

# **4. EXPLORATORY DATA ANALYSIS (EDA)**
Before building the recommendation engine, let's explore our dataset to understand the distribution of touristic places across Egypt and the relationship between user ratings and interaction volume.

In [ ]:
top_locations = places_df['Location'].value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_locations.index, y=top_locations.values, palette="viridis")
plt.title('Top 10 Locations with the Most Touristic Places in Lumeria', fontsize=14)
plt.xlabel('Location / City', fontsize=12)
plt.ylabel('Number of Places', fontsize=12)
plt.xticks(rotation=45)
plt.show()

In [ ]:
place_ratings = df.groupby("Name")["rating"].agg(mean="mean", count="count").reset_index()

plt.figure(figsize=(10, 6))
sns.jointplot(x='mean', y='count', data=place_ratings, kind='scatter', alpha=0.7, color='teal')
plt.suptitle('Relationship between Average Rating and Number of Ratings', y=1.02)
plt.show()

# **5. POPULARITY-BASED RECOMMENDATION MODEL**

**Bayesian Average (Weighted Rating) Formula:**
$$Weighted Rating (WR) = \left(\frac{v}{v+m}\right)R + \left(\frac{m}{v+m}\right)C$$

where,
- **v** is the number of votes for the place;
- **R** is the average rating of the place; 
- **m** is the minimum votes required to be listed in the chart (80th percentile);
- **C** is the mean vote across the whole dataset.

In [ ]:
# Step 1: Calculate average ratings and the number of ratings
place_ratings = df.groupby("Name")["rating"].agg(mean="mean", count="count").reset_index()
place_ratings.sort_values(by=["mean", "count"], ascending=False).head(10)

In [ ]:
new_df = pd.merge(place_ratings, df[["Name", "Location", "Type Of Site"]], on="Name", how="left").sort_values(by=["mean", "count"], ascending=False)
new_df.drop_duplicates(subset=["Name"], inplace=True)
new_df.head()

In [ ]:
C = new_df["mean"].mean()
m = new_df["count"].quantile(0.80)

print(f"Mean vote across all places (C): {C:.2f}")
print(f"Minimum votes required to be listed (m): {m:.2f}")

In [ ]:
def Weighted_Rating(v: float, R: float) -> float:
    return (v * R / (v + m)) + (m * C / (v + m))

In [ ]:
new_df["Weighted Rating"] = new_df.apply(lambda x: Weighted_Rating(v=x["count"], R=x["mean"]), axis=1)

# **6.🏅 TOP 10 PLACES IN EGYPT (OVERALL)**

In [ ]:
final_recommendations = new_df.sort_values(by="Weighted Rating", ascending=False)
final_recommendations[["Name", "Location", "Type Of Site", "count", "mean", "Weighted Rating"]].head(10)

# **7. DYNAMIC CITY-BASED RECOMMENDATIONS**
A custom function designed to return top places filtered by specific cities (e.g., Cairo, Luxor, Aswan)..

In [ ]:
def build_popular_recommender(location_name, percentile=0.80):
    city_df = df[df['Location'].str.contains(location_name, case=False, na=False)]
    
    city_ratings = city_df.groupby("Name")["rating"].agg(mean="mean", count="count").reset_index()
    city_ratings = pd.merge(city_ratings, places_df[["Name", "Location", "Type Of Site"]], on="Name", how="left").drop_duplicates(subset=["Name"])
    
    C = city_ratings['mean'].mean()
    m = city_ratings['count'].quantile(percentile)
    
    qualified_places = city_ratings[city_ratings['count'] >= m].copy()

    qualified_places["Weighted Rating"] = qualified_places.apply(lambda x: Weighted_Rating(v=x["count"], R=x["mean"]), axis=1)

    qualified_places = qualified_places.sort_values(by="Weighted Rating", ascending=False)
    
    return qualified_places[["Name", "Location", "Type Of Site", "count", "mean", "Weighted Rating"]].head(5)

print("🌟 Top 5 Popular Places in Cairo:")
display(build_popular_recommender('Cairo'))

print("\n🌟 Top 5 Popular Places in Luxor:")
display(build_popular_recommender('Luxor'))